# Fitify — Pose-Landmark Exercise Classifier (Kaggle)

Path B: BlazePose landmarks → normalized sequences → small temporal classifier (GRU / 1D-CNN). Self-contained: run top to bottom on a GPU kernel.

**Aspect-ratio fix baked in:** extraction converts MediaPipe's normalized coords to true-aspect pixel space (`x*W, y*H, z*W`) before the hip-center + torso-scale normalization. The dataset is ~20% portrait / 79% landscape, so this removes real training noise (and the portrait-inference bug).

> **App parity:** the deployed pose pipeline must apply the identical `x*W, y*H, z*W` (live camera frame W/H) before normalization, or the bug returns at inference.

## 1. Setup
**Accelerator must be `GPU T4 x2`** (Settings → Accelerator). Kaggle's current PyTorch does **not** support the P100 (sm_60); T4 is sm_75.

In [ ]:
# Kaggle pre-installs torch/numpy/sklearn/opencv; add the rest.
# (no output suppression, so install problems are visible)
!pip install -q mediapipe onnxscript
import mediapipe as mp
print('mediapipe', mp.__version__)
from mediapipe.tasks.python import vision as _v  # verify Tasks vision API
print('tasks.vision OK')
import os
os.makedirs('models', exist_ok=True)
os.makedirs('scripts', exist_ok=True)
open('models/__init__.py', 'w').close()

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Write the pipeline modules

In [ ]:
%%writefile models/dataset.py
"""
Fitify ML — Dataset & Normalization
Loads extracted landmarks, applies size/position-invariant normalization,
builds per-frame feature vectors, and provides a seeded stratified split.

Normalization (the part that matters most):
  1. re-origin every joint to the hip-center (midpoint of L/R hip)
  2. scale by torso length (hip-center -> shoulder-center distance)
This makes features invariant to where the person is in the frame, how far
from the camera, and their absolute body size.

MediaPipe Pose landmark indices used:
  11 L-shoulder, 12 R-shoulder, 23 L-hip, 24 R-hip
"""

import os, json
import numpy as np
import torch
from torch.utils.data import Dataset

L_SH, R_SH, L_HIP, R_HIP = 11, 12, 23, 24
EPS = 1e-6


def normalize_sequence(seq):
    """
    seq: (T, 33, 4) raw landmarks (x,y,z,visibility)
    returns: (T, 33*3) normalized xyz, hip-centered & torso-scaled, per frame.
    Visibility is dropped from features but used for nothing here (already
    filtered upstream); keep xyz only to stay compact.
    """
    xyz = seq[..., :3].copy()                              # (T,33,3)
    hip_center = (xyz[:, L_HIP] + xyz[:, R_HIP]) / 2.0      # (T,3)
    sh_center = (xyz[:, L_SH] + xyz[:, R_SH]) / 2.0         # (T,3)
    torso = np.linalg.norm(sh_center - hip_center, axis=-1, keepdims=True)  # (T,1)
    torso = np.maximum(torso, EPS)
    xyz = xyz - hip_center[:, None, :]                     # re-origin
    xyz = xyz / torso[:, None, :]                          # scale
    return xyz.reshape(xyz.shape[0], -1).astype(np.float32)  # (T, 99)


class LandmarkDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        feats = normalize_sequence(self.X[i])             # (T,99)
        return torch.from_numpy(feats), self.y[i]


def stratified_split(y, seed=42, val=0.15, test=0.15):
    rng = np.random.default_rng(seed)
    idx_train, idx_val, idx_test = [], [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n = len(idx)
        n_test = max(1, int(round(n * test)))
        n_val = max(1, int(round(n * val)))
        idx_test += idx[:n_test].tolist()
        idx_val += idx[n_test:n_test + n_val].tolist()
        idx_train += idx[n_test + n_val:].tolist()
    rng.shuffle(idx_train); rng.shuffle(idx_val); rng.shuffle(idx_test)
    return np.array(idx_train), np.array(idx_val), np.array(idx_test)


def load_splits(data_dir="data", seed=42):
    d = np.load(os.path.join(data_dir, "landmarks.npz"), allow_pickle=True)
    X, y = d["X"], d["y"]
    # angle flag per clip (0=front,1=non-front,2=unclear); default 'unclear'
    # if an older npz without the field is loaded.
    angle = d["angle"] if "angle" in d.files else np.full(len(y), 2, dtype=np.int8)
    tr, va, te = stratified_split(y, seed=seed)
    with open(os.path.join(data_dir, "id_to_label.json")) as fh:
        id_to_label = json.load(fh)
    ds = lambda ix: LandmarkDataset(X[ix], y[ix])
    # test angle flags are returned in the SAME order as test_ds (loader uses
    # shuffle=False), so train.py can split test metrics front vs non-front.
    return ds(tr), ds(va), ds(te), y[tr], id_to_label, np.asarray(angle)[te]


def class_weights(y_train, num_classes):
    counts = np.bincount(y_train, minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    w = counts.sum() / (num_classes * counts)             # inverse-frequency
    return torch.tensor(w, dtype=torch.float32)


In [ ]:
%%writefile models/nets.py
"""
Fitify ML — Models
Two small temporal classifiers over normalized landmark sequences (T, 99).
Both are tiny (~0.1-0.5M params) and export cleanly to ONNX / TFLite.
"""

import torch
import torch.nn as nn


class GRUClassifier(nn.Module):
    """Bi-GRU over the landmark sequence. Default model."""
    def __init__(self, in_dim=99, hidden=128, num_classes=12, layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(in_dim, hidden, num_layers=layers, batch_first=True,
                          bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, num_classes),
        )

    def forward(self, x):                    # x: (B,T,99)
        out, _ = self.gru(x)                 # (B,T,2H)
        feat = out.mean(dim=1)               # temporal mean-pool
        return self.head(feat)


class CNN1DClassifier(nn.Module):
    """1D-CNN over time. Alternative; even smaller, faster."""
    def __init__(self, in_dim=99, num_classes=12, dropout=0.3):
        super().__init__()
        def block(ci, co):
            return nn.Sequential(
                nn.Conv1d(ci, co, 3, padding=1), nn.BatchNorm1d(co),
                nn.ReLU(), nn.MaxPool1d(2))
        self.net = nn.Sequential(block(in_dim, 64), block(64, 128), block(128, 128))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):                    # x: (B,T,99)
        x = x.transpose(1, 2)                # (B,99,T)
        return self.head(self.net(x))


def build_model(name, num_classes, in_dim=99):
    if name == "gru":
        return GRUClassifier(in_dim=in_dim, num_classes=num_classes)
    if name == "cnn":
        return CNN1DClassifier(in_dim=in_dim, num_classes=num_classes)
    raise ValueError(name)


In [ ]:
%%writefile scripts/extract_landmarks.py
"""
Fitify ML — Stage 1: Landmark Extraction
Runs BlazePose (via MediaPipe's Tasks API — PoseLandmarker) over every clip in
the chosen exercise folders and saves a fixed-length landmark sequence per clip.

We use the Tasks API (not the legacy `mp.solutions.pose`, which is absent from
recent MediaPipe builds e.g. 0.10.35 on Kaggle). Same 33-landmark BlazePose
topology. The model bundle is auto-downloaded if not present.

KEY FIX (aspect-ratio): MediaPipe normalizes x by frame WIDTH and y by frame
HEIGHT (different denominators), so its raw normalized space is anisotropic — a
portrait clip and a landscape clip of the same pose come out skewed ~3x on the
x:y axes, and the downstream isotropic torso-scale normalization can't undo it.
We therefore convert to TRUE-ASPECT pixel space here:
    x_px = x_norm * W,   y_px = y_norm * H,   z_px = z_norm * W
(z shares x's width scale). After this, models/dataset.normalize_sequence is
geometrically correct and aspect-invariant.  *** The app's pose pipeline MUST
apply the identical x*W, y*H, z*W before normalization, or the bug returns. ***

Quality filters:
  - visibility: drop clips whose mean landmark visibility < --vis-thresh
  - lighting:   drop clips whose mean frame brightness < --min-brightness
Angle tagging (NOT dropped): a rough front-vs-side heuristic from the
shoulder-width / torso ratio, saved as angle_flag (0=front,1=non-front,2=unclear)
so training can report front vs non-front performance separately.

Output: data/landmarks.npz with arrays
    X      : (N, T, 33, 4) float32  pixel-space (x,y,z) + visibility
    y      : (N,)          int64    class id
    files  : (N,)          str      source filename
    angle  : (N,)          int8     0=front, 1=non-front, 2=unclear
    source : (N,)          str      which dataset root the clip came from
and data/label_map.json / data/id_to_label.json
"""

import os, json, argparse, collections, urllib.request
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# The 12 locked exercises. Folder names are matched case-insensitively /
# whitespace-trimmed against the dataset's actual folder names.
CHOSEN = [
    "squat", "deadlift", "romanian deadlift", "push-up", "pull up",
    "shoulder press", "hammer curl", "lateral raise", "plank",
    "leg raises", "russian twist", "hip thrust",
]
norm = lambda s: s.strip().lower().replace("_", " ")
CHOSEN_N = {norm(c) for c in CHOSEN}

VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv", ".webm"}

# MediaPipe Pose landmark indices
L_SH, R_SH, L_HIP, R_HIP = 11, 12, 23, 24

# Angle heuristic thresholds (rough, tunable): shoulder-width / torso-length.
FRONT_RATIO = 0.45
SIDE_RATIO = 0.22

MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
             "pose_landmarker_full/float16/latest/pose_landmarker_full.task")


def ensure_model(path):
    if not os.path.exists(path):
        print(f"downloading pose model -> {path} ...", flush=True)
        urllib.request.urlretrieve(MODEL_URL, path)
    return path


def make_landmarker(model_path):
    opts = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=model_path),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
    )
    return mp_vision.PoseLandmarker.create_from_options(opts)


def sample_frame_indices(total, num):
    if total <= 0:
        return []
    if total <= num:
        return list(range(total))
    return list(np.linspace(0, total - 1, num).astype(int))


def estimate_angle_flag(arr):
    """arr: (T,33,4) pixel-space. Return 0=front, 1=non-front, 2=unclear."""
    ratios = []
    for fr in arr:
        if fr[L_SH, 3] < 0.5 or fr[R_SH, 3] < 0.5:
            continue
        if fr[L_HIP, 3] < 0.5 and fr[R_HIP, 3] < 0.5:
            continue
        sh_dx = abs(fr[L_SH, 0] - fr[R_SH, 0])
        sh_c = (fr[L_SH, :2] + fr[R_SH, :2]) / 2.0
        hip_c = (fr[L_HIP, :2] + fr[R_HIP, :2]) / 2.0
        torso = float(np.linalg.norm(sh_c - hip_c))
        if torso < 1e-3:
            continue
        ratios.append(sh_dx / torso)
    if len(ratios) < 3:
        return 2
    r = float(np.median(ratios))
    if r >= FRONT_RATIO:
        return 0
    if r <= SIDE_RATIO:
        return 1
    return 2


def extract_clip(fp, landmarker, num_frames):
    """Return (arr (T,33,4) pixel-space, mean_brightness) or (None, 0)."""
    cap = cv2.VideoCapture(fp)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    idxs = sample_frame_indices(total, num_frames)
    frames, brights = [], []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if not ok:
            continue
        h, w = fr.shape[:2]
        brights.append(float(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY).mean()))
        rgb = np.ascontiguousarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = landmarker.detect(mp_image)
        if res.pose_landmarks:
            pts = res.pose_landmarks[0]
            # --- aspect-ratio fix: normalized -> true-aspect pixel space ---
            lm = np.array([[p.x * w, p.y * h, p.z * w, p.visibility]
                           for p in pts], dtype=np.float32)
        else:
            lm = np.zeros((33, 4), dtype=np.float32)
        frames.append(lm)
    cap.release()
    if not frames:
        return None, 0.0
    arr = np.stack(frames, axis=0)
    if arr.shape[0] < num_frames:                       # pad with last frame
        pad = np.repeat(arr[-1:], num_frames - arr.shape[0], axis=0)
        arr = np.concatenate([arr, pad], axis=0)
    return arr[:num_frames], (float(np.mean(brights)) if brights else 0.0)


def discover_classes(roots):
    """Map normalized class name -> list of (root, folder) across all roots."""
    found = collections.defaultdict(list)
    for root in roots:
        if not os.path.isdir(root):
            raise SystemExit(f"dataset root not found: {root}")
        for entry in sorted(os.listdir(root)):
            n = norm(entry)
            if n in CHOSEN_N and os.path.isdir(os.path.join(root, entry)):
                found[n].append((root, entry))
    if not found:
        raise SystemExit(f"No chosen folders found under: {roots}")
    return found


def main(args):
    landmarker = make_landmarker(ensure_model(args.model_path))

    found = discover_classes(args.dataset)
    canon = sorted(found.keys())
    label_map = {name: i for i, name in enumerate(canon)}
    id_to_label = {str(i): name for name, i in label_map.items()}

    X, y, files, angles, sources = [], [], [], [], []
    kept = collections.Counter()
    drop_vis = collections.Counter()
    drop_light = collections.Counter()
    drop_noframe = collections.Counter()
    angle_counts = collections.Counter()      # (cls, flag) -> n

    for cls in canon:
        cid = label_map[cls]
        for root, folder in found[cls]:
            fdir = os.path.join(root, folder)
            vids = [f for f in os.listdir(fdir)
                    if os.path.splitext(f)[1].lower() in VIDEO_EXT]
            src = os.path.basename(root.rstrip("/"))
            print(f"[{cls}] {src}/{folder}: {len(vids)} clips ...", flush=True)
            for f in vids:
                arr, bright = extract_clip(os.path.join(fdir, f), landmarker, args.num_frames)
                if arr is None:
                    drop_noframe[cls] += 1
                    continue
                if bright < args.min_brightness:
                    drop_light[cls] += 1
                    continue
                if float(arr[..., 3].mean()) < args.vis_thresh:
                    drop_vis[cls] += 1
                    continue
                flag = estimate_angle_flag(arr)
                X.append(arr); y.append(cid); files.append(f)
                angles.append(flag); sources.append(src)
                kept[cls] += 1
                angle_counts[(cls, flag)] += 1

    if not X:
        raise SystemExit("No clips survived filtering — loosen thresholds.")

    X = np.stack(X).astype(np.float32)
    y = np.array(y, dtype=np.int64)
    files = np.array(files)
    angles = np.array(angles, dtype=np.int8)
    sources = np.array(sources)

    os.makedirs(args.out, exist_ok=True)
    np.savez_compressed(os.path.join(args.out, "landmarks.npz"),
                        X=X, y=y, files=files, angle=angles, source=sources)
    with open(os.path.join(args.out, "label_map.json"), "w") as fh:
        json.dump(label_map, fh, indent=2)
    with open(os.path.join(args.out, "id_to_label.json"), "w") as fh:
        json.dump(id_to_label, fh, indent=2)

    # ---- summary ----
    print("\n=== extraction summary ===")
    print(f"{'exercise':20}{'kept':>5}{'lowvis':>7}{'dark':>5}{'noframe':>8}"
          f"{'front':>7}{'nonfront':>9}{'unclear':>8}")
    for cls in canon:
        fr0 = angle_counts[(cls, 0)]; fr1 = angle_counts[(cls, 1)]; fr2 = angle_counts[(cls, 2)]
        print(f"{cls:20}{kept[cls]:>5}{drop_vis[cls]:>7}{drop_light[cls]:>5}"
              f"{drop_noframe[cls]:>8}{fr0:>7}{fr1:>9}{fr2:>8}")
    tf = sum(v for (c, f), v in angle_counts.items() if f == 0)
    tn = sum(v for (c, f), v in angle_counts.items() if f == 1)
    tu = sum(v for (c, f), v in angle_counts.items() if f == 2)
    print(f"\nkept {len(X)} | front {tf}  non-front {tn}  unclear {tu}")
    print(f"X shape: {X.shape} (pixel-space xyz + vis)  ->  {args.out}/landmarks.npz")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True, nargs="+",
                    help="one or more dataset roots, each containing class folders")
    ap.add_argument("--out", default="data")
    ap.add_argument("--num-frames", type=int, default=32)
    ap.add_argument("--vis-thresh", type=float, default=0.7)
    ap.add_argument("--min-brightness", type=float, default=50.0)
    ap.add_argument("--model-path", default="pose_landmarker_full.task")
    main(ap.parse_args())


In [ ]:
%%writefile scripts/train.py
"""
Fitify ML — Stage 4: Training
Trains the pose-landmark classifier with class-weighted loss, early stopping
on validation accuracy, then a SINGLE held-out test evaluation producing a
classification report + confusion matrix. Exports best model to ONNX.

Run:
    python scripts/train.py --model gru --epochs 80
"""

import os, sys, json, time, argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from models.dataset import load_splits, class_weights
from models.nets import build_model


def set_seed(s=42):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        ps.append(logits.argmax(1).cpu().numpy())
        ys.append(yb.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    acc = (y == p).mean()
    return acc, y, p


def main(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    train_ds, val_ds, test_ds, y_train, id_to_label, test_angle = load_splits(args.data_dir, args.seed)
    num_classes = len(id_to_label)
    print(f"classes: {num_classes} | train {len(train_ds)} val {len(val_ds)} test {len(test_ds)}")

    # drop_last=True so the CNN's BatchNorm never sees a size-1 final batch.
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size)

    model = build_model(args.model, num_classes).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model: {args.model} | params: {n_params/1e6:.3f}M")

    w = class_weights(y_train, num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    os.makedirs(args.ckpt, exist_ok=True)
    best_val, best_epoch, no_improve = 0.0, 0, 0
    history = []
    t0 = time.time()

    for epoch in range(args.epochs):
        model.train()
        tot, correct, loss_sum = 0, 0, 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(1) == yb).sum().item(); tot += yb.size(0)
        scheduler.step()
        train_acc = correct / tot
        val_acc, _, _ = evaluate(model, val_loader, device)
        history.append({"epoch": epoch+1, "train_acc": train_acc,
                        "val_acc": float(val_acc), "loss": loss_sum/tot})
        print(f"epoch {epoch+1:3d}/{args.epochs} | loss {loss_sum/tot:.4f} "
              f"| train {train_acc:.3f} | val {val_acc:.3f}")

        if val_acc > best_val:
            best_val, best_epoch, no_improve = val_acc, epoch+1, 0
            torch.save(model.state_dict(), os.path.join(args.ckpt, "best.pt"))
        else:
            no_improve += 1
            if no_improve >= args.patience:
                print(f"early stop @ epoch {epoch+1} (best val {best_val:.3f} @ {best_epoch})")
                break

    # ---- one-time held-out test eval ----
    model.load_state_dict(torch.load(os.path.join(args.ckpt, "best.pt")))
    test_acc, y_true, y_pred = evaluate(model, test_loader, device)
    print(f"\n=== TEST accuracy: {test_acc:.4f} (best val {best_val:.4f} @ epoch {best_epoch}) ===")

    labels = [id_to_label[str(i)] for i in range(num_classes)]
    os.makedirs("results", exist_ok=True)

    # classification report + confusion matrix (sklearn if available, else manual)
    try:
        from sklearn.metrics import classification_report, confusion_matrix
        rep = classification_report(y_true, y_pred, target_names=labels, digits=3)
        cm = confusion_matrix(y_true, y_pred)
    except Exception:
        rep = "(install scikit-learn for full report)"
        cm = np.zeros((num_classes, num_classes), int)
        for t, p in zip(y_true, y_pred):
            cm[t, p] += 1
    print("\n" + (rep if isinstance(rep, str) else ""))
    print("confusion matrix (rows=true, cols=pred):")
    print("      " + " ".join(f"{i:>4}" for i in range(num_classes)))
    for i, row in enumerate(cm):
        print(f"{i:>2} {labels[i][:14]:14} " + " ".join(f"{v:>4}" for v in row))

    # ---- test accuracy split by camera angle (front vs non-front) ----
    angle_lines = ["\nangle breakdown (test set):"]
    names = {0: "front", 1: "non-front", 2: "unclear"}
    for flag, nm in names.items():
        m = test_angle == flag
        if m.sum() == 0:
            angle_lines.append(f"  {nm:10} n=0")
            continue
        a = (y_true[m] == y_pred[m]).mean()
        angle_lines.append(f"  {nm:10} n={int(m.sum()):4d}  acc={a:.4f}")
    angle_report = "\n".join(angle_lines)
    print(angle_report)

    with open("results/test_report.txt", "w") as fh:
        fh.write(f"test_acc {test_acc:.4f}\nbest_val {best_val:.4f} @ {best_epoch}\n\n")
        fh.write(str(rep) + "\n\nlabels: " + ", ".join(labels) + "\n")
        fh.write(angle_report + "\n")
    with open("results/history.json", "w") as fh:
        json.dump(history, fh, indent=2)
    np.save("results/confusion_matrix.npy", cm)

    # ---- ONNX export (portable: cloud server or convert to TFLite) ----
    model.eval().cpu()
    T = args.num_frames
    dummy = torch.randn(1, T, 99)
    onnx_path = os.path.join(args.ckpt, "fitify_pose.onnx")
    try:
        torch.onnx.export(model, dummy, onnx_path,
                          input_names=["landmarks"], output_names=["logits"],
                          dynamic_axes={"landmarks": {0: "batch", 1: "frames"},
                                        "logits": {0: "batch"}},
                          opset_version=17, dynamo=False)
        size_mb = os.path.getsize(onnx_path) / 1e6
        print(f"\nexported ONNX -> {onnx_path} ({size_mb:.2f} MB)")
    except Exception as e:
        print(f"\n[warn] ONNX export skipped: {e}")
        print("       (pip install onnx onnxscript to enable; best.pt is still saved)")
    print(f"total time: {time.time()-t0:.0f}s")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="data")
    ap.add_argument("--ckpt", default="checkpoints")
    ap.add_argument("--model", default="gru", choices=["gru", "cnn"])
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--patience", type=int, default=15)
    ap.add_argument("--num-frames", type=int, default=32)
    ap.add_argument("--seed", type=int, default=42)
    main(ap.parse_args())


## 3. Extract landmarks (verified_data only)
Point these at your uploaded Kaggle dataset. Two roots = the two verified sources; extraction merges same-named class folders across both.

In [ ]:
# Dataset mount paths (verified_data, both sources):
ROOT_BTC   = '/kaggle/input/datasets/vishardmehta/fitify-private-dataset/verified_data/verified_data/data_btc_10s'
ROOT_CRAWL = '/kaggle/input/datasets/vishardmehta/fitify-private-dataset/verified_data/verified_data/data_crawl_10s'
assert os.path.isdir(ROOT_BTC) and os.path.isdir(ROOT_CRAWL), \
    'paths wrong — run:  !find /kaggle/input -type d -name data_btc_10s -o -type d -name data_crawl_10s'
!python scripts/extract_landmarks.py --dataset "$ROOT_BTC" "$ROOT_CRAWL" \
    --num-frames 32 --vis-thresh 0.7 --min-brightness 50

## 4. Train GRU

In [ ]:
!python scripts/train.py --model gru --epochs 120 --patience 20
import shutil, os
os.makedirs('results/gru', exist_ok=True)
for f in ['test_report.txt','confusion_matrix.npy','history.json']:
    shutil.copy(f'results/{f}', f'results/gru/{f}')
shutil.copy('checkpoints/fitify_pose.onnx', 'results/gru/fitify_pose_gru.onnx')

## 5. Train CNN

In [ ]:
!python scripts/train.py --model cnn --epochs 120 --patience 20
os.makedirs('results/cnn', exist_ok=True)
for f in ['test_report.txt','confusion_matrix.npy','history.json']:
    shutil.copy(f'results/{f}', f'results/cnn/{f}')
shutil.copy('checkpoints/fitify_pose.onnx', 'results/cnn/fitify_pose_cnn.onnx')

## 6. Compare — confusion matrices, angle split, recommendation

In [ ]:
import numpy as np, json, matplotlib.pyplot as plt
labels = [v for k, v in sorted(json.load(open('data/id_to_label.json')).items(), key=lambda x:int(x[0]))]

def show(tag, ax):
    cm = np.load(f'results/{tag}/confusion_matrix.npy')
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(tag.upper()); ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7); ax.set_yticklabels(labels, fontsize=7)
    return cm

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for tag, ax in zip(['gru','cnn'], axes):
    show(tag, ax)
plt.tight_layout(); plt.show()

for tag in ['gru','cnn']:
    print('='*60, tag.upper()); print(open(f'results/{tag}/test_report.txt').read())

In [ ]:
# Most-confused pairs + recommendation
for tag in ['gru','cnn']:
    cm = np.load(f'results/{tag}/confusion_matrix.npy').astype(float)
    off = cm.copy(); np.fill_diagonal(off, 0)
    pairs = sorted(((off[i,j], labels[i], labels[j]) for i in range(len(labels)) for j in range(len(labels)) if off[i,j]>0), reverse=True)[:5]
    acc = np.trace(cm)/cm.sum()
    print(f'\n{tag.upper()} test_acc={acc:.3f}  top confusions:')
    for n,a,b in pairs: print(f'   {a} -> {b}: {int(n)}')


## 7. Ship decision
Pick the model with the better **macro-F1** (read from the reports above); if GRU and CNN are within ~1%, ship the **CNN** (smaller + faster inference). The exported ONNX is in `results/<model>/`. Feed the same normalized 99-dim landmark sequence to the form-feedback module — it's already hip-centered & torso-scaled.